<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 1 (Lunes) — Predicción — Series de tiempo**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 📉 Series de tiempo (ARIMA): tasa de desempleo en Estados Unidos

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Entender cómo ha variado el desempleo en EE. UU. y qué se quiere pronosticar (la tasa de los próximos años). |
| 🔍 Comprensión de los datos | Cargamos `output.csv` (tasas por condado, 1990-2016), graficamos la serie y revisamos distribución, ACF y PACF. |
| 🧹 Preparación de los datos | Promediamos por mes/año para obtener una observación por periodo y descomponemos con STL. |
| 🤖 Modelado | Buscamos los parámetros (p, d, q) × (P, D, Q) con criterio AIC y ajustamos el modelo ARIMA/SARIMA. |
| 📊 Evaluación | Revisamos coeficientes, errores y residuos para decidir si el modelo es adecuado. |
| 🚀 Despliegue y conclusiones | Pronosticamos los próximos periodos y discutimos las limitaciones (varianza creciente). |

> 🎯 **Analogía:** una serie de tiempo es el electrocardiograma de un fenómeno: tiene un ritmo base (tendencia), latidos que se repiten (estacionalidad) y ruido; ARIMA aprende ese ritmo para anticipar el siguiente latido.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 👉 **1** | Lunes | **Predicción: regresión lineal y series de tiempo** |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('output.csv', 'dia_2')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

---
## 🎯 Fase 1 · Comprensión del negocio

Entender cómo ha variado el desempleo en EE. UU. y qué se quiere pronosticar (la tasa de los próximos años).

📚 Lectura recomendada: [3] [5]

# Serie de tiempo
Una serie temporal o cronológica es una secuencia de datos, observaciones o valores, medidos en determinados momentos y ordenados cronológicamente. Los datos pueden estar espaciados a intervalos iguales (como la temperatura en un observatorio meteorológico en días sucesivos al mediodía) o desiguales (como el peso de una persona en sucesivas mediciones en el consultorio médico, la farmacia, etc.)

### Autores
   * Efrén Jiménez

## Análisis del Problema

##### Breve historia del empleo en los Estados Unidos.

Durante la década de 1940, el Departamento de Trabajo de los EE. UU., Específicamente la Oficina de Estadísticas Laborales (BLS), comenzó a recopilar información sobre el empleo a través de encuestas de hogares mensuales. Otras series de datos están disponibles hasta 1912. La tasa de desempleo varió desde un 1% durante la Primera Guerra Mundial hasta un 25% durante la Gran Depresión (en la mayoría de los países comenzó en 1929 y duró hasta 1941). la depresión más larga, profunda y extendida del siglo XX). Más recientemente, alcanzó un máximo de 10.8% en noviembre de 1982 y 10.0% en octubre de 2009. El desempleo tiende a aumentar durante las recesiones y cae durante las expansiones. De 1948 a 2015, el desempleo promedió alrededor del 5,8%. Los Estados Unidos han experimentado 11 recesiones desde el final de la posguerra en 1948.

![Serie de tiempo real (manchas solares) con su ACF e histograma](https://raw.githubusercontent.com/alkaline-ml/pmdarima/master/doc/img/sunspots/untransformed.png)
*Figura: Serie de tiempo real (manchas solares) con su ACF e histograma. [9]*

##### Algunas definiciones básicas

La Oficina de Estadísticas Laborales de los Estados Unidos ha definido los conceptos básicos de empleo de la siguiente manera:

- Las personas con empleo son * empleadas *.
- Las personas que están desempleadas, en busca de empleo y disponibles para trabajar están * desempleadas *.
- Las personas que no están empleadas ni desempleadas no están en la fuerza laboral.

¿Quién no está en la fuerza laboral?

- La fuerza laboral está formada por personas empleadas y desempleadas que tienen 16 años o más.
- El resto, aquellos que no tienen trabajo y no están buscando uno, se cuentan como "no en la fuerza laboral". Muchos de los que no están en la fuerza laboral van a la escuela o están jubilados. Las responsabilidades familiares mantienen a otros fuera de la fuerza laboral.
- Las personas confinadas a instituciones como las prisiones o en servicio activo en las Fuerzas Armadas están excluidas de los cálculos de la fuerza laboral.

#####  Tasas de desempleo

Hay una variedad de medidas utilizadas para rastrear el estado del mercado laboral de los Estados Unidos. Cada uno proporciona una visión de los factores que afectan el empleo. A continuación se presentan algunos de los más comunes:

- Siempre hay algo de desempleo, con personas que cambian de trabajo y nuevos participantes en la fuerza laboral que buscan trabajo. Esto se conoce como desempleo friccional. Por este motivo, la Reserva Federal apunta a la ** tasa natural de desempleo o NAIRU **, que fue de alrededor del 5% en 2015.
- ** La tasa de desempleo (U-3) **, medida como el número de personas desempleadas dividida por la fuerza laboral civil, aumentó de 5.0% en diciembre de 2007 a un máximo de 10.0% en octubre de 2009, antes de caer al nivel bajo del 4,7% en mayo de 2016. Esta medida excluye a las personas que no están en la fuerza laboral, lo que puede distorsionar su interpretación si un gran número de personas en edad de trabajar se desanime y deja de buscar trabajo.
- ** La tasa de desempleo (U-6) ** es una medida más amplia del desempleo, que trata a los trabajadores adicionales como desempleados (por ejemplo, los empleados a tiempo parcial por razones económicas, etc.). La tasa U-6 aumentó de 8.8% en diciembre de 2007 a un máximo de 17.1% en noviembre de 2009, antes de caer constantemente a 9.7% en mayo de 2016.

#####  Resumen de los datos utilizados

Los datos utilizados a los fines de este informe representan las estadísticas de desempleo en el área local desde enero de 1990 hasta diciembre de 2016, desglosadas por estado y mes. La versión formateada de los datos en formato CSV para los fines de este análisis se obtuvo de [Kaggle] (https://www.kaggle.com/jayrav13/unemployment-by-county-us). Los datos sin formato sin procesar están disponibles aquí en el [sitio web de la Oficina de Estadísticas Laborales de los Estados Unidos] (https://data.bls.gov/map/MapToolServlet?survey=la&map=county&seasonal=u).

** Nota: ** Estas tasas de desempleo son tasas generales mensuales U-3 y no están ajustadas por temporada ni categorizadas por edad, sexo, nivel de educación, etc. Tenga en cuenta que en realidad, las tasas de desempleo que se reportan generalmente se ajustan por estacionalidad. (El ajuste estacional es una técnica estadística que intenta medir y eliminar las influencias de patrones estacionales predecibles para revelar cómo el empleo y el desempleo cambian de un mes a otro).

## librerías

In [ ]:
%pip install -q "pmdarima>=2.1.1"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import itertools
import pandas as pd
import numpy as np
from scipy.stats import norm
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")
from pmdarima.arima import auto_arima
from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm

import plotly.graph_objs as go
import plotly.io as pio


---
## 🔍 Fase 2 · Comprensión de los datos

Cargamos `output.csv` (tasas por condado, 1990-2016), graficamos la serie y revisamos distribución, ACF y PACF.

📚 Lectura recomendada: [3] [6]

## Entendimiento de los Datos

Con el fin de analizar y predecir el desempleo en Estados Unidos se utilizan **885 548 observaciones y 5 variables** (1990-2016):

- `Year`: numérica (año, 1990-2016).
- `Month`: categórica ordinal (nombre del mes).
- `State`: categórica (estado).
- `County`: categórica (condado).
- `Rate`: numérica (tasa de desempleo, %).

In [ ]:
# Lee el archivo CSV y almacena los datos en un dataframe
unemp_data = pd.read_csv('sample_data/output.csv')

# Obtener información general sobre el conjunto de datos
print(unemp_data.info())

# Tasa de desempleo anual para los Estados Unidos desde 1990 hasta 2016
annual_rate = unemp_data.groupby('Year')['Rate'].mean()
#years = [1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1999, 2010, 2011, 2012, 2013, 2014, 2015, 2016]

# Crear un gráfico de línea de la tasa de desempleo anual promedio
plt.plot(annual_rate.index, annual_rate)
plt.title("Tasa media anual de desempleo en Estados Unidos")
plt.xlabel("Año")
plt.ylabel("Tasa de desempleo")
plt.show()

Este conjunto de datos abarca más de 3 recesiones. El país pasó por una recesión en 1990. El colapso de la burbuja de Internet y los ataques del 11 de septiembre empujaron al país a otra recesión en 2001, poco después de que George W. Bush asumiera el cargo. La crisis financiera de 2007 a 2009 dio lugar a una tercera recesión. Podemos ver el aumento correspondiente en las tasas de desempleo para estos períodos de tiempo en el gráfico anterior, el más marcado es el que muestra la reciente crisis financiera.

---
## 🧹 Fase 3 · Preparación de los datos

Promediamos por mes/año para obtener una observación por periodo y descomponemos con STL.

📚 Lectura recomendada: [6]

### Preparación del conjunto de datos para su análisis.
Este conjunto de datos tiene casi 900 mil observaciones con aproximadamente 3 mil observaciones por cada mes de cada año desde 1990 hasta 2016. A continuación, encontremos la tasa mensual promedio de desempleo para cada uno de los 27 años. Tendremos 12 observaciones por cada año de esta manera. Crearemos un conjunto de datos abreviado con estas tasas medias mensuales.

In [ ]:
#Las primeras filas de este conjunto de datos
print(unemp_data.head())
print(unemp_data.info())

Exiten 324 observaciones en este nuevo conjunto de datos. Hemos reducido de casi 3k observaciones por mes por año a una observación por mes por año al tomar el promedio mensual. Este conjunto de datos ahora está listo para su posterior análisis.

### Trabajando con series de tiempo

R contiene varias clases de series de tiempo. En este caso, trabajaremos con la clase `ts` básica. Una clase poderosa de `zoo` proporciona más capacidades. El paquete `xts` agrega extensibilidad a la clase` zoo`. Muchas otras clases de series de tiempo R están disponibles.

Los siguientes paquetes son necesarios para su posterior análisis:

---
## 🔍 Fase 2 · Comprensión de los datos — visualización, estacionariedad y autocorrelación

Cargamos `output.csv` (tasas por condado, 1990-2016), graficamos la serie y revisamos distribución, ACF y PACF.

Ahora creamos la serie de tiempo de la tasa nacional de desempleo a partir del conjunto anterior (en `pandas`, una `Series` indexada por año).

In [ ]:
#Gráfico the time series
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.plot(annual_rate)
plt.title('Gráfico de series de tiempo de la tasa de desempleo en los Estados Unidos')
plt.show()

In [ ]:
#Gráfico the time series
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.plot(np.log(annual_rate))
plt.title('Gráfico de series de tiempo de la tasa de desempleo en los Estados Unidos')
plt.show()

Decimos que una serie de tiempo con una variación constante es **estacionaria**. Las propiedades de una serie temporal estacionaria no varían con el tiempo.

Existe un patrón **estacional** cuando una serie está influenciada por factores estacionales (por ejemplo, el trimestre del año, el mes o el día de la semana). La estacionalidad es siempre *de un período fijo y conocido*. Por lo tanto, las series temporales estacionales a veces se denominan **periódicas** series temporales.

Existe un patrón **cíclico** cuando los datos muestran subidas y caídas que *no son de período fijo*. Por ejemplo, los ciclos económicos que suelen durar varios años, pero de antemano se desconoce la duración del ciclo actual.

Existe una **tendencia** cuando hay un aumento o disminución a largo plazo en los datos. No tiene que ser lineal. A veces nos referiremos a una tendencia que "cambia de dirección" cuando podría ir de una tendencia creciente a una tendencia decreciente.

En el gráfico anterior, notamos las siguientes propiedades de esta serie de tiempo.

- Esta serie temporal tiene una tendencia significativa.
- Tiene un componente estacional.
- Tiene un patrón cíclico mostrado por el aumento y la caída de las tasas de desempleo correspondientes a las recesiones y expansiones.
- La duración del ciclo (en años) varía con el tiempo.
- El gráfico de la serie logarítmica transformada parece tener un tamaño ligeramente más uniforme para el componente estacional que el no transformado. Así que vamos a utilizar la serie de log transformado para un análisis más detallado

Cualquier serie temporal con una tendencia implica que varía con el tiempo, lo que significa que no es estacionario**.

Una serie temporal estacionaria sin patrón estacional o cíclico y sin tendencia perceptible se dice que es **independiente distribuida de manera idéntica** o **i.i.d.** An i.i.d. Las series de tiempo tienen una distribución normal con una distribución en forma de campana y una gráfica Q-Q de línea recta. Tal serie de tiempo también se llama ruido blanco.

Vamos a trazar la distribución de nuestras series de tiempo a continuación:

In [ ]:
# Gráfico la distribución - histograma y QQ de la serie temporal
def dist_ts(df, col='residual', bins=40):
    fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 4))

    temp = np.array(df).ravel()
    breaks = np.linspace(temp.min(), temp.max(), bins+1)

    ax1.hist(temp, bins=breaks, density=True)
    ax1.set_title(f"Distribution of {col}")
    ax1.set_xlabel(col)

    sm.qqplot(temp, line='s', ax=ax2)
    ax2.set_title(f"Q-Q Normal plot of {col}")

    plt.show()

dist_ts(np.log(annual_rate), col='US Unemp Rate')

![Autocorrelación (ACF)](https://raw.githubusercontent.com/alkaline-ml/pmdarima/master/doc/img/lynx_autocorr.png)
*Figura: ACF de una serie con ciclos regulares; las barras fuera de la banda de confianza indican autocorrelación significativa. [9]*

Las líneas de puntos azules en los gráficos ACF y PACF son los intervalos de confianza del 95% para los valores de correlación. Las líneas verticales muestran la correlación. Cualquier línea vertical que cruce las líneas punteadas azules indica un valor de correlación significativo. Un i.i.d. las series de tiempo normalmente distribuidas no tendrán valores significativos de ACF o PACF más allá del retraso 0. En este caso, el gráfico de ACF tiene varios retrasos significativos no cero y muestra cierto patrón de aumento y caída. También el ACF está disminuyendo o decayendo lentamente. La gráfica PACF muestra valores significativos para más de un retraso, lo que indica que podría haber una tendencia. Esto claramente no es una serie temporal estacionaria.

El proceso de extracción de la tendencia, el ciclo y los componentes estacionales se denomina descomposición. A continuación, veremos los métodos de descomposición.

**Descomposición basada en tasas de cambio**

Esta es una técnica importante para todos los tipos de análisis de series de tiempo, especialmente para el ajuste estacional. Busca construir, a partir de una serie temporal observada, una serie de componentes (que podrían usarse para reconstruir el original mediante adiciones o multiplicaciones) donde cada una de ellas tiene una cierta característica o tipo de comportamiento. Por ejemplo, las series de tiempo generalmente se descomponen en:

$ T_t $, el componente de tendencia en el tiempo t, que refleja la progresión a largo plazo de la serie.

$ C_t $, el componente cíclico en el momento t, que describe fluctuaciones repetidas pero no periódicas.

$ S_t $, el componente estacional en el tiempo t, que refleja la estacionalidad.

$ I_t $, el componente irregular (o "ruido") en el tiempo t, que describe influencias aleatorias e irregulares. Esta es la parte de la serie que no se puede atribuir a los componentes estacionales, de ciclo o de tendencia y se conoce como residual, resto o error. Por lo tanto, una serie temporal que utiliza un modelo aditivo se puede considerar como:

$ y_t $ = $ T_t $ + $ C_t $ + $ S_t $ + $ I_t $

mientras que, un modelo multiplicativo sería:

$ y_t $ = $ T_t $ x $ C_t $ x $ S_t $ x $ I_t $

Se usaría un modelo aditivo cuando las variaciones en torno a la tendencia no varíen con el nivel de las series temporales, mientras que como modelo multiplicativo sería apropiado si la tendencia es proporcional al nivel de las series temporales.

### Descomposición de STL

A veces, la tendencia y los componentes cíclicos se agrupan en uno, llamado componente de tendencia-ciclo. El componente de ciclo de tendencia se puede denominar simplemente componente de "tendencia", aunque puede contener un comportamiento cíclico. Por ejemplo, una descomposición estacional de series de tiempo, conocida como **S** easonal & **T** rend **descomposición** utilizando **L** oess o **STL**, descompone una serie de tiempo en componentes estacionales, de tendencia e irregulares utilizando loess y grafican los componentes por separado, por lo que el componente cíclico (si está presente en los datos) se incluye en el gráfico del componente de "tendencia".

Como su nombre indica, este modelo hace lo siguiente:

- La tendencia se elimina utilizando un modelo de regresión LOESS.
- El componente estacional se elimina mediante una regresión en componentes periódicos.
- El resto se conoce como el residual.

A continuación, aplicaremos el modelo de descomposición STL a nuestras series de tiempo transformadas de registro y veremos si esto elimina efectivamente la tendencia y la estacionalidad.

In [ ]:
# Función para la descomposición STL de una serie de tiempo en componentes
def ts_decomp(df, col='Series Name', span=0.13, mult=True):
    if mult:
        temp = np.log(df)
    else:
        temp = df

    spans = int(span * len(temp))
    decomposed = sm.tsa.seasonal_decompose(temp, model='additive', period=12, extrapolate_trend='freq')

    fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(10, 8))
    ax1.plot(temp, label=col)
    ax1.set_title('Original')
    ax2.plot(decomposed.trend, label='Trend')
    ax2.set_title('Trend')
    ax3.plot(decomposed.seasonal, label='Seasonal')
    ax3.set_title('Seasonal')
    ax4.plot(decomposed.resid, label='Residual')
    ax4.set_title('Residual')
    plt.suptitle(f'Decomposition of {col} with loess span = {span}')
    plt.tight_layout()
    plt.show()

    return decomposed.resid

us_unemp_decomp = ts_decomp(annual_rate, col='US Unemp Rate', mult=False)

Si el proceso STL ha eliminado la tendencia y los patrones con éxito, la distribución del resto o residual debe ser i.i.d. con ACF y PACF que no muestran retrasos significativos distintos de cero. De los trazados de ACF y PACF del residuo, todavía vemos algunos valores significativos de retardo no cero. Así que el proceso STL no ha eliminado la tendencia o los patrones completamente.

## Modelos ARIMA para la serie residual.

Ahora que hemos investigado las propiedades básicas de las series de tiempo y el método de descomposición de STL, investiguemos los modelos para tratar los residuos. Trabajaremos con lo que se conoce y los modelos **modelo autorregresivo integrado de promedio móvil** o **ARIMA**. En su núcleo, estos modelos son solo casos especiales de regresión lineal.

### Modelo Autoregresivo

Los valores de una serie de tiempo **autoregrisiva** o **AR** se determinan mediante una combinación lineal de los valores pasados. En otras palabras, el modelo AR representa la correlación serial en los valores de la serie de tiempo. Podemos escribir el valor de una serie autorregresiva o **ordenar p** o **AR (p)** series en el momento t como:

$$ x_t = \ alpha_1 x_ {t-1} + \ alpha_2 x_ {t-2} \ dots \ alpha_p x_ {t-p} + w_t $$

Un proceso AR tiene las siguientes propiedades:

- $ \ rho_0 = 1 $ siempre.
- $ p_k = \ alpha ^ k $
- Número de valores de PACF distintos de cero = p.

Los modelos AR son específicamente para **series temporales estacionarias**. Si la varianza no es constante o el componente de tendencia no se ha eliminado, los modelos AR no producirán resultados estadísticos.

### Modelo de media móvil

Para un modelo **de media móvil** o **MA**, el valor de la serie de tiempo en el tiempo `t` está determinado por una combinación lineal de términos de ruido blanco pasados. En otras palabras, el modelo de MA explica la correlación de series en términos de ruido. Podemos escribir el modelo MA (q) como la combinación lineal de los últimos términos de ruido blanco `q` $ w_i $:

$$ x_t = w_t + \ beta_1 w_ {t-1} + \ beta_2 w_ {t-2} + \ cdots + \ beta_q w_ {t-q} $$

Un proceso de MA tiene las siguientes propiedades:

- $ \ rho_0 = 1 $ siempre. (Auto correlación en el retraso 0)
- Número de $ \ rho_k distinto de cero; k \ ne 0 $ valores = q. (auto correlación en el retraso k)

Los modelos MA son específicamente para **series de tiempo estacionarias**. Si la varianza no es constante o el componente de tendencia no se ha eliminado, los modelos MA no producirán resultados estadísticos.

### El modelo de media móvil autoregresivo

Podemos combinar los modelos AR y MA para crear un **modelo de media móvil autorregresiva** o **ARMA**. Este modelo explica la correlación serial tanto en términos de ruido como en valores. Podemos escribir un modelo ARMA de orden `(p, q)` como:

$$ x_t = \ alpha_1 x_ {t-1} + \ alpha_2 x_ {t-2} \ dots \ alpha_p x_ {t-p}
w_t + \ beta_1 w_ {t-1} + \ beta_2 w_ {t-2} + \ cdots + \ beta_q w_ {t-q} $$

### Modelo de media móvil integrada autoregresiva

El modelo **de media móvil integrada autorregresiva**, o **ARIMA** agrega un término integrador al modelo ARMA. El componente de integración realiza la diferenciación para modelar un componente de paseo aleatorio. El componente integrante modela una de las **partes no estacionarias** de una serie de tiempo. El modelo ARIMA está definido por órdenes p, d, q. El orden del operador de diferenciación del término integrador está definido por `d`.

Ahora, intentemos estimar los parámetros ARIMA (p, d, q) de la serie de tiempo (recuerde, esta es una serie de tiempo no estacionaria) e imprimimos un resumen del mejor modelo y trazamos el ACF y el PACF del residual.

---
## 🤖 Fase 4 · Modelado

Buscamos los parámetros (p, d, q) × (P, D, Q) con criterio AIC y ajustamos el modelo ARIMA/SARIMA.

📚 Lectura recomendada: [4] [8] [9]

In [ ]:
p = d = q = range(0, 2)
pdq = list(itertools.product(p, d, q))
seasonal_pdq = [(x[0], x[1], x[2], 12) for x in list(itertools.product(p, d, q))]
print('Examples of parameter combinations for Seasonal ARIMA...')
print('SARIMAX: {} x {}'.format(pdq[1], seasonal_pdq[1]))
print('SARIMAX: {} x {}'.format(pdq[1], seasonal_pdq[2]))
print('SARIMAX: {} x {}'.format(pdq[2], seasonal_pdq[3]))
print('SARIMAX: {} x {}'.format(pdq[2], seasonal_pdq[4]))

In [ ]:
for param in pdq:
  for param_seasonal in seasonal_pdq:
    try:
        mod = sm.tsa.statespace.SARIMAX(annual_rate,
                                        order=param,
                                        seasonal_order=param_seasonal,
                                        enforce_stationarity=False,
                                        enforce_invertibility=False)
        results = mod.fit()
        print('ARIMA{}x{}12 - AIC:{}'.format(param, param_seasonal, results.aic))
    except:
        continue

**De los resultados anteriores notamos que:**

El modelo ARIMA (3,0,4) con los siguientes coeficientes tiene el menor AIC (Criterios de información de Akaike)

**Coeficientes:**

| - | ar1 | ar2 | ar3 | ma1 | ma2 | ma3 | ma4 |
| - | -------- | ------- | -------- | -------- | -------- | ---- ---- | -------- |
| - | -1.0076 | 0,2590 | 0.7093 | 1.8248 | 1.3592 | 0.5135 | 0.2456 |
| es decir. | 0.0557 | 0.0958 | 0.0563 | 0.0696 | 0,1589 | 0.1603 | 0.0702 |

sigma ^ 2 estimado como 0.0006552: probabilidad de registro = 725.9, aic = -1435.8

- Todos los coeficientes son significativos (el error estándar asociado con estos coeficientes es algunas veces más pequeño que el valor del coeficiente en sí)
- El $ \ sigma ^ 2 $ o la varianza de la serie residual es un orden de magnitud menor que la serie original.

---
## 📊 Fase 5 · Evaluación

Revisamos coeficientes, errores y residuos para decidir si el modelo es adecuado.

📚 Lectura recomendada: [3]

> ⚠️ **Nota:** parte del texto interpretativo de este cuaderno proviene de una versión en R; al ejecutar en Python los números exactos (AIC, coeficientes, RMSE) pueden variar ligeramente. Use las salidas de sus propias celdas para interpretar.

### Evaluación

El paquete R `forecast` en particular, tiene herramientas para estimar automáticamente los modelos ** ARIMA ** estacionales y utilizar estos modelos para hacer pronósticos futuros.

La función `auto.arima` del paquete` forecast` hace lo siguiente:

- Realiza una regresión progresiva progresiva a partir del orden máximo de cada coeficiente para encontrar el modelo ARIMA estacional óptimo. La opimización se realiza utilizando los Criterios de Información de Akaike (AIC) y los Criterios de Información Bayesiana (BIC).
- El modelo ARIMA estacional puede escribirse como ARIMA (p, d, q) (P, D, Q). Ya hemos explorado el modelo estándar de ARIMA, ARIMA (p, d, q). Los términos estacionales adicionales modelan el comportamiento estacional y de tendencia:
  - P es el orden del modelo autorregresivo estacional.
  - D es el orden de un operador diferenciador o integrador, que modela la tendencia.
  - Q es el orden del modelo de media móvil estacional.

Los ** Criterios de Información Bayesiana ** o ** BIC ** están estrechamente relacionados con los Criterios de Información de Akaike. El BIC pesa el número de parámetros en el modelo por el registro del número de observaciones.

Esta es la parte interesante. Utilizaremos `auto.arima` para ajustar un modelo lineal a nuestra serie de tiempo mensual de tasa de desempleo nacional y lo usaremos para pronosticar las tasas de desempleo futuras, es decir, lo usaremos para pronosticar las tasas de desempleo de enero a diciembre de 2017. Ya tenemos las tasas reales o observadas disponibles para enero a julio de 2017 en [sitio web de BLS] (https://data.bls.gov/timeseries/LNU04000000). Podemos comparar las tasas pronosticadas con las tasas reales para ver si nuestro modelo es adecuado o no.

In [ ]:
# Fit an ARIMA model using auto_arima
fit_US_Unemp = auto_arima(annual_rate, seasonal=False, suppress_warnings=True, error_action='ignore')

# Print the model summary
print(fit_US_Unemp.summary())

In [ ]:
# fit ARIMA model
arima_model = ARIMA(annual_rate.reset_index(drop=True), order=(1, 0, 0))
arima_fit = arima_model.fit()

De estos resultados notamos los siguientes puntos:

- El modelo estimado es ARIMA (1,1,4) (0,0,2).
- Sigma ^ 2 de la serie original = 0.046737234933687
- sigma ^ 2 del modelo estimado = 0.001906
- Los errores estándar de los coeficientes del modelo son generalmente un orden de magnitud menor que los valores de los coeficientes del modelo. Todos los coeficientes de este modelo parecen ser significativos
- Se producen una serie de métricas de error que incluyen:
  - Error medio, ME.
  - Error cuadrático medio, RMSE.
  - Error absoluto medio, MAE.
  - Error de predicción media, MPE, error de predicción absoluta media, MAPE y error de cuadratura absoluta media, o MASE. El error de predicción se calcula actualizando iterativamente el modelo en una ventana en expansión y midiendo el error de predecir el siguiente valor más allá de la ventana. Tenga en cuenta que el ** MAPE es bastante alto ** en comparación con los otros errores.
  - ACF1 es la correlación del modelo residual en el primer desfase distinto de cero, o retardo 1, que no es significativo en este caso.

Usaremos el modelo óptimo estimado para hacer predicciones en el futuro.

In [ ]:
# Graficar la serie de tiempo original y el ajuste del modelo
plt.plot(annual_rate.index, annual_rate.values, label='Serie de tiempo original')
plt.plot(annual_rate.index, arima_fit.fittedvalues, label='Ajuste del modelo ARIMA')

plt.title('Ajuste del modelo ARIMA a la serie de tiempo')
plt.xlabel('Año')
plt.ylabel('Tasa de desempleo')
plt.legend()
plt.show()

La línea azul en el gráfico anterior muestra las tasas pronosticadas y el frotis gris denota los intervalos de confianza.

Notamos que el 80% y el 95% de los intervalos de confianza son casi proporcionales al valor pronosticado en sí mismo y se hacen cada vez más amplios a medida que avanza el tiempo. Esto implica que la varianza puede estar aumentando con el tiempo, o la heterocedasticidad. Eso significa que esto puede no ser adecuado para nuestros datos de desempleo.

A continuación, comparemos los valores pronosticados con los valores reales (no ajustados por estacionalidad) obtenidos del sitio web de BLS en paralelo:

---
## 🚀 Fase 6 · Despliegue y conclusiones

Pronosticamos los próximos periodos y discutimos las limitaciones (varianza creciente).

📚 Lectura recomendada: [3]

In [ ]:
# Make a forecast for the next 5 steps
steps=10
forecast = arima_fit.forecast(steps=steps)

forecast.index=range(annual_rate.index[-1], annual_rate.index[-1]+steps)
# Graficar la serie de pronósticos
plt.plot(annual_rate)
plt.plot(forecast)
plt.legend(['Actual', 'Forecast'])
plt.show()

In [ ]:
forecast

Esto muestra que a medida que avanzamos en el tiempo, los valores pronosticados van más y más lejos de la marca, lo que implica que esto no es un buen ajuste para nuestros datos.

### Resumen

Analizamos la tasa de desempleo nacional de los EE. UU. (U-3, no ajustada por estacionalidad), originalmente condensada del sitio web de BLS, como una serie de tiempo univariable de 1990 a 2016 y utilizamos los métodos ARIMA para predecir las tasas para 2017. Realizamos las siguientes observaciones:

- La serie de tiempo no es estacionaria.
- Tiene un componente estacional y una tendencia y un componente cíclico.
- Utilizamos la descomposición de STL para eliminar la tendencia y los componentes estacionales de la serie y encontramos que un modelo ARIMA (3,0,4) es un buen ajuste para el resto de la serie.
- Utilizamos la función `auto.arima` del paquete` forecast` para encontrar el modelo ARIMA más adecuado y lo usamos para pronosticar la tasa de desempleo en 2017.
- Realizamos una comparación visual del valor pronosticado y los valores reales obtenidos del [sitio web de BLS] (https://data.bls.gov/timeseries/LNU04000000) de enero a julio de 2017 y encontramos que los valores pronosticados se alejan aún más de los valores reales a medida que avanza el tiempo, lo que implica que este modelo no es un buen ajuste

### Conclusión

Nuestro objetivo fue estudiar las tasas de desempleo U-3 no ajustadas estacionalmente como una serie de tiempo univariada y encontrar modelos lineales que describieran las variaciones de manera efectiva. STL descompuso la serie de tiempo y ajustamos un modelo ARIMA (p, d, q) al resto. También ajustamos un modelo ARIMA (p, d, q) (P, D, Q) y lo usamos para pronosticar las tasas de desempleo para 2017. Después de comparar algunos de los valores reales con los valores pronosticados, llegamos a la conclusión de que este modelo no es bueno. ajuste.

En realidad, existen varios modelos que podríamos explorar e incluir otros factores o características para modelar nuestra tasa de desempleo y ayudarnos a hacer un mejor pronóstico, pero eso está fuera del alcance de este análisis.

---
## 🧪 Refuerzo: prueba de estacionariedad (ADF) y diferenciación

ARIMA necesita una serie **estacionaria**: media y varianza que no cambian con el tiempo. La prueba **Dickey-Fuller aumentada (ADF)** parte de la hipótesis nula "la serie **no** es estacionaria": si el *valor p* < 0.05, se rechaza y la serie se considera estacionaria [7]. Si no lo es, se **diferencia** ($y_t-y_{t-1}$), que es justo el parámetro $d$ de ARIMA.

In [ ]:
from statsmodels.tsa.stattools import adfuller

def reporte_adf(serie, nombre):
    est, p, *_ = adfuller(serie.dropna())
    print(f'{nombre:28s} ADF = {est:7.3f} | valor p = {p:.4f} -> ' + ('estacionaria' if p < 0.05 else 'NO estacionaria'))

reporte_adf(annual_rate, 'Serie original')
reporte_adf(annual_rate.diff(), 'Primera diferencia (d = 1)')

### ✍️ Ejercicios
1. ¿Con qué valor de $d$ la serie pasa la prueba ADF? ¿Coincide con el $d$ que eligió `auto_arima`?
2. Grafique `annual_rate.diff()` y compárela con la serie original.

---
## ✅ Conclusiones y siguientes pasos

Pronosticar una serie de tiempo exige **estacionariedad** (o volverla estacionaria), un buen modelo de autocorrelación y mirar con cuidado los **intervalos de confianza**: si crecen con el horizonte, la predicción a largo plazo pierde valor [3]. El **Martes (Día 2)** pasa de predecir números a **clasificar** con árboles de decisión.

---
## 📚 Referencias

Fuentes adicionales para profundizar: [10], [11].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3.ª ed.). OTexts. https://otexts.com/fpp3/

**[4]** Box, G. E. P., Jenkins, G. M., Reinsel, G. C., & Ljung, G. M. (2015). *Time Series Analysis: Forecasting and Control* (5.ª ed.). Wiley.

**[5]** U.S. Bureau of Labor Statistics. (s. f.). *Local Area Unemployment Statistics (LAUS)*. https://www.bls.gov/lau/

**[6]** Cleveland, R. B., Cleveland, W. S., McRae, J. E., & Terpenning, I. (1990). STL: A seasonal-trend decomposition procedure based on loess. *Journal of Official Statistics, 6*(1), 3–73.

**[7]** Dickey, D. A., & Fuller, W. A. (1979). Distribution of the estimators for autoregressive time series with a unit root. *Journal of the American Statistical Association, 74*(366a), 427–431. https://doi.org/10.1080/01621459.1979.10482531

**[8]** Seabold, S., & Perktold, J. (2010). statsmodels: Econometric and statistical modeling with Python. *Proceedings of the 9th Python in Science Conference*, 92–96. https://doi.org/10.25080/Majora-92bf1922-011

**[9]** Smith, T. G., et al. (2017–). *pmdarima: ARIMA estimators for Python* [Software]. https://alkaline-ml.com/pmdarima

**[10]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

**[11]** Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55